# Fine-tune PaddleOCR `rec_mv3_none_bilstm_ctc` cho manga tiếng Nhật (Kaggle H100)

Notebook này dành cho **Kaggle H100** và pipeline Manga109-s:

1. Cài môi trường PaddleOCR
2. Tải Manga109-s từ Hugging Face (Kaggle Secret)
3. Parse XML + crop text box để tạo REC dataset
4. Tạo dictionary tiếng Nhật từ label
5. Tải pretrained `rec_mv3_none_bilstm_ctc_v2.0_train`
6. Sinh config train theo kiến trúc `rec_mv3_none_bilstm_ctc.yml`
7. Train / resume / export inference model

In [ ]:
import os, sys, json, random, zipfile, subprocess
import xml.etree.ElementTree as ET
from typing import Optional
from PIL import Image
from tqdm import tqdm

In [ ]:
# Tránh conflict từ stack torch preinstalled (notebook này chỉ dùng Paddle)
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y',
    'torch', 'torchvision', 'torchaudio', 'triton',
    'fastai', 'fastcore', 'fastdownload', 'nbdev'
], check=False)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'paddlepaddle-gpu==3.0.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'])

import paddle
print('Paddle version:', paddle.__version__)
print('CUDA available:', paddle.device.is_compiled_with_cuda())
print('GPU count:', paddle.device.cuda.device_count())
for i in range(paddle.device.cuda.device_count()):
    print(f'GPU {i}:', paddle.device.cuda.get_device_name(i))

In [ ]:
PADDLEOCR_DIR = '/kaggle/working/PaddleOCR'
if not os.path.isdir(PADDLEOCR_DIR):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git {PADDLEOCR_DIR}
!pip install -q -r {PADDLEOCR_DIR}/requirements.txt
print('PaddleOCR ready:', PADDLEOCR_DIR)

## Data từ Hugging Face + Kaggle Secret

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

secret_label = 'HF_TOKEN'
secret_value = UserSecretsClient().get_secret(secret_label)

WORK_DIR         = '/kaggle/working'
HF_DOWNLOAD_DIR  = f'{WORK_DIR}/manga109s_dataset'
EXTRACT_DIR      = f'{WORK_DIR}/manga109_extracted'
DATASET_ROOT     = f'{EXTRACT_DIR}/Manga109s_released_2023_12_07'
DATA_DIR         = f'{WORK_DIR}/train_data/manga109_rec_h100'
OUTPUT_DIR       = f'{WORK_DIR}/output/rec_manga109_h100_mv3_bilstm_ctc'
PRETRAINED_DIR   = f'{WORK_DIR}/pretrained'
DICT_PATH        = f'{WORK_DIR}/manga109_jp_dict.txt'
CONFIG_PATH      = f'{WORK_DIR}/rec_manga109_h100_mv3_bilstm_ctc.yml'

for p in [HF_DOWNLOAD_DIR, EXTRACT_DIR, DATA_DIR, OUTPUT_DIR, PRETRAINED_DIR]:
    os.makedirs(p, exist_ok=True)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])

login(secret_value)
zip_path = f'{HF_DOWNLOAD_DIR}/Manga109s_released_2023_12_07.zip'

if not os.path.isfile(zip_path):
    snapshot_download(
        repo_id='hal-utokyo/Manga109-s',
        repo_type='dataset',
        local_dir=HF_DOWNLOAD_DIR,
        token=secret_value,
    )

if not os.path.isdir(DATASET_ROOT) or not os.listdir(DATASET_ROOT):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(EXTRACT_DIR)

print('Dataset root:', DATASET_ROOT)
print('Content:', os.listdir(DATASET_ROOT))

In [ ]:
class Manga109RecDatasetCreator:
    def __init__(self, dataset_root: str):
        self.dataset_root = dataset_root
        self.annotations_dir = os.path.join(dataset_root, 'annotations')
        self.images_dir = os.path.join(dataset_root, 'images')

    def create_rec_dataset(
        self,
        output_dir: str,
        min_width: int = 12,
        min_height: int = 12,
        max_samples: Optional[int] = 120_000,
        train_ratio: float = 0.92,
        seed: int = 42,
    ):
        random.seed(seed)

        train_img_dir = os.path.join(output_dir, 'train')
        val_img_dir = os.path.join(output_dir, 'val')
        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(val_img_dir, exist_ok=True)

        train_gt = os.path.join(output_dir, 'rec_gt_train.txt')
        val_gt = os.path.join(output_dir, 'rec_gt_val.txt')

        xml_files = sorted(f for f in os.listdir(self.annotations_dir) if f.endswith('.xml'))
        print(f'Found {len(xml_files)} books. Parsing samples...')
        all_samples = []

        for xml_file in tqdm(xml_files, desc='Parsing books'):
            book_title = os.path.splitext(xml_file)[0]
            xml_path = os.path.join(self.annotations_dir, xml_file)
            try:
                root = ET.parse(xml_path).getroot()
            except Exception:
                continue

            for page in root.findall('.//page'):
                page_idx = page.get('index')
                if page_idx is None:
                    continue
                img_filename = f'{int(page_idx):03d}.jpg'
                img_path = os.path.join(self.images_dir, book_title, img_filename)
                if not os.path.exists(img_path):
                    continue
                try:
                    img = Image.open(img_path).convert('RGB')
                except Exception:
                    continue

                for text_obj in page.findall('text'):
                    try:
                        xmin = int(text_obj.get('xmin'))
                        ymin = int(text_obj.get('ymin'))
                        xmax = int(text_obj.get('xmax'))
                        ymax = int(text_obj.get('ymax'))
                        label = (text_obj.text or '').strip().replace('\n', '').replace('\t', ' ')
                        if not label:
                            continue
                        if (xmax - xmin) < min_width or (ymax - ymin) < min_height:
                            continue
                        all_samples.append((img.crop((xmin, ymin, xmax, ymax)), label))
                    except Exception:
                        continue

            if max_samples and len(all_samples) >= max_samples * 2:
                break

        random.shuffle(all_samples)
        if max_samples:
            all_samples = all_samples[:max_samples]

        split_idx = int(len(all_samples) * train_ratio)
        train_samples = all_samples[:split_idx]
        val_samples = all_samples[split_idx:]
        print(f'Split: {len(train_samples):,} train | {len(val_samples):,} val')

        def _write(samples, img_dir, gt_path, split, start_idx=1):
            with open(gt_path, 'w', encoding='utf-8') as f:
                for i, (crop, label) in enumerate(tqdm(samples, desc=f'Saving {split}'), start=start_idx):
                    img_name = f'word_{i:07d}.png'
                    p = os.path.join(img_dir, img_name)
                    try:
                        crop.save(p)
                    except Exception:
                        continue
                    f.write(f'{split}/{img_name}\t{label}\n')
            return len(samples)

        n_train = _write(train_samples, train_img_dir, train_gt, 'train', 1)
        n_val = _write(val_samples, val_img_dir, val_gt, 'val', n_train + 1)
        return n_train, n_val, train_gt, val_gt

train_label = os.path.join(DATA_DIR, 'rec_gt_train.txt')
val_label = os.path.join(DATA_DIR, 'rec_gt_val.txt')

if not (os.path.isfile(train_label) and os.path.isfile(val_label)):
    creator = Manga109RecDatasetCreator(DATASET_ROOT)
    n_train, n_val, train_label, val_label = creator.create_rec_dataset(output_dir=DATA_DIR)
    print('Created dataset:', n_train, n_val)
else:
    print('Dataset exists, skip crop.')

In [ ]:
def build_dict_from_labels(*label_files):
    chars = set()
    max_len = 0
    for path in label_files:
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.rstrip('\n').split('\t', 1)
                if len(parts) < 2:
                    continue
                txt = parts[1]
                chars.update(txt)
                max_len = max(max_len, len(txt))
    return sorted(chars), max_len

chars, max_text_len = build_dict_from_labels(train_label, val_label)
with open(DICT_PATH, 'w', encoding='utf-8') as f:
    for c in chars:
        f.write(c + '\n')

print('Dictionary chars:', len(chars))
print('Max text length:', max_text_len)
print('Saved:', DICT_PATH)
MAX_TEXT_LENGTH = min(max_text_len + 8, 96)

In [ ]:
PRETRAINED_URL = 'https://paddleocr.bj.bcebos.com/dygraph_v2.0/en/rec_mv3_none_bilstm_ctc_v2.0_train.tar'
PRETRAINED_TAR = os.path.join(PRETRAINED_DIR, 'rec_mv3_none_bilstm_ctc_v2.0_train.tar')
PRETRAINED_ROOT = os.path.join(PRETRAINED_DIR, 'rec_mv3_none_bilstm_ctc_v2.0_train')
PRETRAINED_PATH = os.path.join(PRETRAINED_ROOT, 'best_accuracy')

if not os.path.isfile(PRETRAINED_TAR):
    !wget -q --show-progress -O {PRETRAINED_TAR} {PRETRAINED_URL}
if not os.path.isdir(PRETRAINED_ROOT):
    !tar -xf {PRETRAINED_TAR} -C {PRETRAINED_DIR}
print('Pretrained path:', PRETRAINED_PATH)
!ls -lh {PRETRAINED_ROOT}

In [ ]:
import yaml

def count_lines(path):
    with open(path, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

n_train = count_lines(train_label)
n_val = count_lines(val_label)
N_GPU = max(1, paddle.device.cuda.device_count())

# H100: batch size lớn hơn để tăng throughput
if N_GPU == 1:
    BATCH_PER_CARD = 512
else:
    BATCH_PER_CARD = 384

STEPS_PER_EPOCH = max(1, n_train // (BATCH_PER_CARD * N_GPU))
EVAL_STEP = max(500, STEPS_PER_EPOCH * 2)

latest_ckpt = os.path.join(OUTPUT_DIR, 'latest')
resume_ckpt = latest_ckpt if os.path.isfile(latest_ckpt + '.pdparams') else ''

cfg = {
    'Global': {
        'use_gpu': True,
        'epoch_num': 120,
        'log_smooth_window': 20,
        'print_batch_step': 20,
        'save_model_dir': OUTPUT_DIR,
        'save_epoch_step': 10,
        'eval_batch_step': [0, EVAL_STEP],
        'cal_metric_during_train': True,
        'pretrained_model': '' if resume_ckpt else PRETRAINED_PATH,
        'checkpoints': resume_ckpt,
        'save_inference_dir': f'{OUTPUT_DIR}/inference',
        'use_visualdl': False,
        'character_dict_path': DICT_PATH,
        'max_text_length': MAX_TEXT_LENGTH,
        'infer_mode': False,
        'use_space_char': True,
        'distributed': N_GPU > 1,
        'save_res_path': f'{OUTPUT_DIR}/predicts.txt',
        'seed': 2026,
    },
    'Optimizer': {
        'name': 'Adam',
        'beta1': 0.9,
        'beta2': 0.999,
        'lr': {'name': 'Cosine', 'learning_rate': 0.0008, 'warmup_epoch': 2},
        'regularizer': {'name': 'L2', 'factor': 1.0e-5},
    },
    'Architecture': {
        'model_type': 'rec',
        'algorithm': 'CRNN',
        'Transform': None,
        'Backbone': {'name': 'MobileNetV3', 'scale': 0.5, 'model_name': 'large'},
        'Neck': {'name': 'SequenceEncoder', 'encoder_type': 'rnn', 'hidden_size': 96},
        'Head': {'name': 'CTCHead', 'fc_decay': 0},
    },
    'Loss': {'name': 'CTCLoss'},
    'PostProcess': {'name': 'CTCLabelDecode'},
    'Metric': {'name': 'RecMetric', 'main_indicator': 'acc'},
    'Train': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': DATA_DIR,
            'label_file_list': [train_label],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'RecAug': None},
                {'CTCLabelEncode': None},
                {'RecResizeImg': {'image_shape': [3, 32, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label', 'length']}},
            ],
        },
        'loader': {
            'shuffle': True,
            'batch_size_per_card': BATCH_PER_CARD,
            'drop_last': True,
            'num_workers': 8,
            'use_shared_memory': True,
        },
    },
    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': DATA_DIR,
            'label_file_list': [val_label],
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'CTCLabelEncode': None},
                {'RecResizeImg': {'image_shape': [3, 32, 320]}},
                {'KeepKeys': {'keep_keys': ['image', 'label', 'length']}},
            ],
        },
        'loader': {
            'shuffle': False,
            'drop_last': False,
            'batch_size_per_card': BATCH_PER_CARD,
            'num_workers': 8,
            'use_shared_memory': True,
        },
    },
}

with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print('Train samples:', n_train)
print('Val samples:', n_val)
print('GPUs:', N_GPU, 'batch/card:', BATCH_PER_CARD)
print('Config saved:', CONFIG_PATH)
print(open(CONFIG_PATH, 'r', encoding='utf-8').read())

In [ ]:
if N_GPU > 1:
    gpu_list = ','.join(str(i) for i in range(N_GPU))
    cmd = f'python -m paddle.distributed.launch --gpus="{gpu_list}" {PADDLEOCR_DIR}/tools/train.py -c {CONFIG_PATH}'
else:
    cmd = f'python {PADDLEOCR_DIR}/tools/train.py -c {CONFIG_PATH}'

print('Train command:')
print(cmd)
!{cmd}

In [ ]:
best_model = os.path.join(OUTPUT_DIR, 'best_accuracy')
print('Best model exists:', os.path.isfile(best_model + '.pdparams'))
!ls -lh {OUTPUT_DIR} | sed -n '1,120p'

export_cmd = f'python {PADDLEOCR_DIR}/tools/export_model.py -c {CONFIG_PATH} -o Global.pretrained_model={best_model} Global.save_inference_dir={OUTPUT_DIR}/inference'
print(export_cmd)
!{export_cmd}
!ls -lh {OUTPUT_DIR}/inference